# 特征工程：生成 cleaned.csv

只使用两份原始CSV。删除核心信息无效的比赛；保留已有Elo，缺失时查找严格早于比赛日的最近评分，没有时长上限。用全部有效赛果重算此前3/5场积分，再筛选双方Elo完整、最近5场完整、三项普通赔率有效的比赛，删除已确定的23列。

从上到下运行，输出cleaned.csv和简短处理汇总。

## 1. 读取文件和处理规则

In [1]:
from pathlib import Path
from bisect import bisect_left
from collections import defaultdict, deque
import json
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data/raw/Matches.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/raw/Matches.csv').exists(), '请从项目根目录或notebooks目录运行。'
CORE = ['Division', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FTHome', 'FTAway', 'FTResult']
MISSING_TOKENS = {'', 'nan', 'na', 'n/a', 'null', 'none'}
FORM_COLUMNS = [f'Form{k}{side}' for side in ['Home', 'Away'] for k in [3, 5]]
COUNTRY = {d: d for d in 'ARG AUT BRA CHN DEN FIN IRL JAP MEX NOR POL ROM RUS SUI SWE USA'.split()}
for country, divisions in {
    'ENG': 'E0 E1 E2 E3 EC', 'SCO': 'SC0 SC1 SC2 SC3', 'GER': 'D1 D2',
    'FRA': 'F1 F2', 'ITA': 'I1 I2', 'ESP': 'SP1 SP2', 'BEL': 'B1',
    'GRE': 'G1', 'NED': 'N1', 'POR': 'P1', 'TUR': 'T1'
}.items():
    COUNTRY.update({d: country for d in divisions.split()})

def normalize(value):
    return ' '.join(unicodedata.normalize('NFKC', str(value)).split())



def read_raw(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    for col in frame:
        missing = frame[col].str.strip().str.lower().isin(MISSING_TOKENS)
        frame.loc[missing, col] = np.nan
    return frame

## 2. 删除核心信息缺失或无效的比赛

任何一项核心字段缺失即删除，不推导缺失赛果。检查重复和同队同日多场，出现冲突时停止并核对。

In [2]:
def prepare_matches(raw):
    df = raw.copy()
    df.insert(0, 'SourceRow', np.arange(2, len(df) + 2))
    reasons = pd.Series('', index=df.index, dtype=object)
    missing = df[CORE].isna().any(axis=1)
    reasons.loc[missing] = 'missing_core_field'
    for col in ['Division', 'HomeTeam', 'AwayTeam']:
        df[col] = df[col].map(lambda v: normalize(v) if pd.notna(v) else v)
    day = pd.to_datetime(df.MatchDate, format='%Y-%m-%d', errors='coerce')
    reasons.loc[reasons.eq('') & day.isna()] = 'invalid_date'
    for col in ['FTHome', 'FTAway']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    scores = df[['FTHome', 'FTAway']]
    ok = np.isfinite(scores).all(axis=1) & scores.ge(0).all(axis=1) & scores.mod(1).eq(0).all(axis=1)
    expected = np.where(df.FTHome > df.FTAway, 'H', np.where(df.FTHome < df.FTAway, 'A', 'D'))
    reasons.loc[reasons.eq('') & ~ok] = 'invalid_score'
    reasons.loc[reasons.eq('') & df.FTResult.ne(expected)] = 'result_score_conflict'
    reasons.loc[reasons.eq('') & df.HomeTeam.eq(df.AwayTeam)] = 'same_home_and_away_team'
    excluded = raw.loc[reasons.ne('')].copy()
    excluded.insert(0, 'SourceRow', df.loc[reasons.ne(''), 'SourceRow'])
    excluded['Reason'] = reasons.loc[reasons.ne('')]
    df = df.loc[reasons.eq('')].copy()
    df['Country'] = df.Division.map(COUNTRY)
    if df.Country.isna().any():
        raise ValueError('发现未知联赛代码，需先补充国家映射。')
    df['MatchDate'] = day.loc[df.index]
    df[['FTHome', 'FTAway']] = df[['FTHome', 'FTAway']].astype(int)
    key = ['Division', 'MatchDate', 'HomeTeam', 'AwayTeam']
    if df.duplicated(key).any():
        raise ValueError('发现重复比赛，需核对后再生成，不自动合并。')
    df = df.sort_values(['MatchDate', 'SourceRow']).reset_index(drop=True)
    appearances = pd.concat([
        df[['Country', 'MatchDate', 'HomeTeam']].rename(columns={'HomeTeam': 'Team'}),
        df[['Country', 'MatchDate', 'AwayTeam']].rename(columns={'AwayTeam': 'Team'})])
    if appearances.duplicated(['Country', 'MatchDate', 'Team']).any():
        raise ValueError('发现同队同日多场记录，需核对后再生成。')
    return df, excluded

## 3. 填补缺失的 Elo

仅补缺失的一侧，保留原值。每次填补记录来源和评分日期。日期必须严格小于比赛日，没有历史时长限制。

完成查找后，删除任意一方 Elo 仍缺失的比赛。最终文件要求双方评分都有值。

In [3]:
def elo_lookup(raw):
    ratings = raw.copy()
    ratings['date'] = pd.to_datetime(ratings.date, format='%Y-%m-%d', errors='coerce')
    ratings['elo'] = pd.to_numeric(ratings.elo, errors='coerce')
    good = ratings[['date', 'club', 'country']].notna().all(axis=1) & np.isfinite(ratings.elo)
    ratings = ratings.loc[good].copy()
    for col in ['club', 'country']:
        ratings[col] = ratings[col].map(normalize)
    counts = ratings.groupby(['country', 'club', 'date']).elo.nunique()
    conflicts = set(counts[counts > 1].index)
    if conflicts:
        ratings = ratings.loc[[tuple(r) not in conflicts for r in ratings[['country', 'club', 'date']].itertuples(index=False, name=None)]]
    ratings = ratings.drop_duplicates(['country', 'club', 'date']).sort_values('date')
    lookup = {key: (list(group.date), group.elo.to_numpy(float))
              for key, group in ratings.groupby(['country', 'club'], sort=False)}
    return lookup, {'invalid_rating_rows': int((~good).sum()), 'conflicting_team_dates': len(conflicts)}

def find_earlier_rating(lookup, key, day):
    if key not in lookup:
        return np.nan, pd.NaT, 'team_not_found'
    dates, values = lookup[key]
    pos = bisect_left(dates, day) - 1
    if pos < 0:
        return np.nan, pd.NaT, 'no_strictly_earlier_rating'
    return float(values[pos]), dates[pos], 'filled'

def fill_elo(df, lookup):
    df = df.copy()
    for side in ['Home', 'Away']:
        col = side + 'Elo'
        original = pd.to_numeric(df[col], errors='coerce')
        original = original.where(np.isfinite(original))
        missing = original.isna()
        values = original.to_numpy(copy=True)
        sources = np.where(missing, 'missing', 'Matches.csv').astype(object)
        source_dates = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
        status = np.where(missing, '', 'original_kept').astype(object)
        for i in np.flatnonzero(missing.to_numpy()):
            key = (df.at[i, 'Country'], df.at[i, side + 'Team'])
            value, rating_date, reason = find_earlier_rating(lookup, key, df.at[i, 'MatchDate'])
            status[i] = reason
            if reason == 'filled':
                values[i] = value
                sources[i] = 'EloRatings.csv'
                source_dates.at[i] = rating_date
        df[col] = values
        df[col + 'OriginallyMissing'] = missing.to_numpy()
        df[col + 'Source'] = sources
        df[col + 'RatingDate'] = source_dates
        df[col + 'FillStatus'] = status
        # Date restrictions only concern newly filled ratings; original dates are unknown.
        filled = df[col + 'Source'].eq('EloRatings.csv')
        assert (df.loc[filled, col + 'RatingDate'] < df.loc[filled, 'MatchDate']).all()
    return df

def retain_complete_elo(df):
    """最终样本只保留双方Elo都存在的比赛；不改变已建立的历史池。"""
    valid = np.isfinite(df[['HomeElo', 'AwayElo']]).all(axis=1)
    removed = df.loc[~valid].copy()
    removed['Reason'] = 'missing_elo_after_fill'
    retained = df.loc[valid].reset_index(drop=True).copy()
    return retained, removed

## 4. 重算此前3场、5场积分

先生成当天全部赛前统计，再更新当天赛果；只有完整窗口才输出积分总和。

先用全部有效赛果计算历史，再过滤最终样本，避免因历史某场缺 Elo 而丢失其已知比分。

In [4]:
def rebuild_form(df):
    """最近恰好3/5场的积分总和；仅使用严格早于本场日期的比赛。"""
    df = df.copy()
    totals = {col: np.full(len(df), np.nan) for col in FORM_COLUMNS}
    observed = {col: np.zeros(len(df), dtype=np.int64) for col in FORM_COLUMNS}
    prior_counts = {side: np.zeros(len(df), dtype=np.int64) for side in ['Home', 'Away']}
    history = defaultdict(lambda: deque(maxlen=5))
    counts = defaultdict(int)
    for day, batch in df.groupby('MatchDate', sort=True):
        # 先读取当天全部赛前历史，再更新当天赛果。
        for r in batch.itertuples():
            for side in ['Home', 'Away']:
                key = (r.Country, getattr(r, side + 'Team'))
                prior = list(history[key])
                assert all(date < day for date, points in prior)
                prior_counts[side][r.Index] = counts[key]
                for k in [3, 5]:
                    col = f'Form{k}{side}'
                    n = min(k, len(prior))
                    observed[col][r.Index] = n
                    if n == k:
                        totals[col][r.Index] = sum(points for date, points in prior[-k:])
        for r in batch.itertuples():
            for side, win in [('Home', 'H'), ('Away', 'A')]:
                key = (r.Country, getattr(r, side + 'Team'))
                points = 1 if r.FTResult == 'D' else 3 if r.FTResult == win else 0
                history[key].append((day, points))
                counts[key] += 1
    for col in FORM_COLUMNS:
        df[col] = pd.array(totals[col], dtype='Int64')
        df[col + 'Count'] = observed[col]
        df[col + 'Missing'] = np.isnan(totals[col])
    for side in ['Home', 'Away']:
        df[side + 'PriorMatchCount'] = prior_counts[side]
    return df

## 5. 清理、补全并计算此前状态

In [5]:
raw = read_raw(ROOT / 'data/raw/Matches.csv')
elo_raw = read_raw(ROOT / 'data/raw/EloRatings.csv')
history_pool, core_excluded = prepare_matches(raw)
lookup, _ = elo_lookup(elo_raw)
history_pool = fill_elo(history_pool, lookup)
history_pool = rebuild_form(history_pool)
completed, elo_excluded = retain_complete_elo(history_pool)
odds = completed[['OddHome', 'OddDraw', 'OddAway']].apply(pd.to_numeric, errors='coerce')
completed['OddsAvailable'] = np.isfinite(odds).all(axis=1) & odds.gt(1).all(axis=1)

## 6. 筛选共同样本并保存 cleaned.csv

双方须有完整最近5场，且三项普通赔率均为有限数并大于1。未使用的半场、技术统计和原有Elo的评分日期允许为空。

In [6]:
# 用户确定的删除名单：17列暂不使用字段 + 6列填补记录。
DROP_UNUSED_COLUMNS = [
    'MatchTime',
    'MaxHome', 'MaxDraw', 'MaxAway',
    'Over25', 'Under25', 'MaxOver25', 'MaxUnder25',
    'HandiSize', 'HandiHome', 'HandiAway',
    'C_LTH', 'C_LTA', 'C_VHD', 'C_VAD', 'C_HTB', 'C_PHB',
]
DROP_AUDIT_COLUMNS = [
    'HomeEloOriginallyMissing', 'AwayEloOriginallyMissing',
    'HomeEloSource', 'AwayEloSource',
    'HomeEloFillStatus', 'AwayEloFillStatus',
]
DROP_COLUMNS = DROP_UNUSED_COLUMNS + DROP_AUDIT_COLUMNS

form5_ok = completed[['Form5Home', 'Form5Away']].notna().all(axis=1) & completed[['Form5HomeCount', 'Form5AwayCount']].eq(5).all(axis=1)
keep = form5_ok & completed.OddsAvailable
cleaned = completed.loc[keep].drop(columns=DROP_COLUMNS).reset_index(drop=True).copy()
cleaned_path = ROOT / 'data/processed/cleaned.csv'
cleaned_path.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(cleaned_path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
summary = {
    'input_rows': len(raw), 'valid_history_rows': len(history_pool),
    'core_excluded_rows': len(core_excluded), 'elo_excluded_rows': len(elo_excluded),
    'incomplete_form5_rows': int((~form5_ok).sum()),
    'invalid_odds_rows_after_form5_filter': int((form5_ok & ~completed.OddsAvailable).sum()),
    'rows_after': len(cleaned), 'columns_after': len(cleaned.columns),
    'removed_columns': DROP_COLUMNS,
}
report_dir = ROOT / 'reports/feature_engineering'
report_dir.mkdir(parents=True, exist_ok=True)
(report_dir / 'cleaned_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
readme = f"""# cleaned 数据集

[cleaned.csv](cleaned.csv)：{len(cleaned):,}场、{cleaned.shape[1]}列。只用原始两份CSV直接生成，完整处理代码见[02_features.ipynb](../../notebooks/02_features.ipynb)。

已有Elo保留；缺失时查找同一国家和球队、严格早于比赛日的最近评分，没有年限上限。双方Elo仍有缺失、最近5场不足或普通赔率无效的比赛删除。Form为此前恰好3/5场的积分总和，历史使用全部核心赛果有效的比赛。

## 删除列

{chr(10).join('- ' + col for col in DROP_COLUMNS)}

## 保留列

{chr(10).join('- ' + col for col in cleaned.columns)}

半场、技术统计及原有Elo的评分日期仍允许缺失。表中含身份、赛果和辅助记录，不代表所有列都进入模型；本场赛后统计不作为赛前预测输入。
"""
(ROOT / 'data/processed/cleaned.README.md').write_text(readme, encoding='utf-8')
print(f'已生成cleaned.csv：{len(cleaned):,}场，{cleaned.shape[1]}列。')
display(pd.DataFrame([summary]).drop(columns='removed_columns'))

已生成cleaned.csv：133,512场，46列。


,input_rows,valid_history_rows,core_excluded_rows,elo_excluded_rows,incomplete_form5_rows,invalid_odds_rows_after_form5_filter,rows_after,columns_after
0,230557,230554,3,92951,2101,1990,133512,46
